In [ ]:
import os
import glob
import re
import datetime
import pandas as pd
import numpy as np
import cv2
from skimage import io, exposure, filters, morphology, measure

# Folder Configuration

INPUT_FOLDER = r"*"   # <--- Update based on file locations
OUTPUT_FOLDER = r"" # <--- Update based on where you want the output folder to be generated.

# Tunable parameters
TUMOR_GAMMA = 0.5
MIN_TUMOR_SIZE = 10000
HOLE_SIZE = 1000

# Align the centroid
def align_mask_by_centroids(mask_src, center_src, shape_dest, center_dest):
    """
    Pastes mask_src into a canvas of shape_dest, aligning the centroids.
    Handles different image sizes (cropping/padding) automatically.
    """

    mask_aligned = np.zeros(shape_dest, dtype=bool)
    

    dy = int(center_dest[0] - center_src[0])
    dx = int(center_dest[1] - center_src[1])
    
    src_h, src_w = mask_src.shape
    dst_h, dst_w = shape_dest
    
    dst_y1 = max(0, dy)
    dst_y2 = min(dst_h, dy + src_h)
    dst_x1 = max(0, dx)
    dst_x2 = min(dst_w, dx + src_w)
    
    src_y1 = max(0, -dy)
    src_y2 = src_y1 + (dst_y2 - dst_y1)
    src_x1 = max(0, -dx)
    src_x2 = src_x1 + (dst_x2 - dst_x1)
    
    if dst_y2 > dst_y1 and dst_x2 > dst_x1:
        mask_aligned[dst_y1:dst_y2, dst_x1:dst_x2] = \
            mask_src[src_y1:src_y2, src_x1:src_x2]
            
    return mask_aligned

# Name parsing for date and time

def parse_filename(filename):
    # Regex for YYYY_MM_DD__HH_MM_SS
    match = re.search(r'(\d{4}_\d{2}_\d{2}__\d{2}_\d{2}_\d{2})', filename)
    if match:
        time_str = match.group(1)
        dt_object = datetime.datetime.strptime(time_str, "%Y_%m_%d__%H_%M_%S")
        # Well Name is everything before the date
        well_name = filename.split(time_str)[0].strip('_-.')
        return well_name, dt_object
    return None, None

# Segmentation
def get_tumor_mask(img_stack):
    # Handle Channels
    if img_stack.ndim == 3 and img_stack.shape[0] < 5:
        raw_red = img_stack[0, :, :]
    elif img_stack.ndim == 3:
        raw_red = img_stack[:, :, 0]
    else:
        raw_red = img_stack
        
    # Segment
    red_enhanced = exposure.adjust_gamma(raw_red, gamma=TUMOR_GAMMA)
    try: thresh = filters.threshold_li(red_enhanced)
    except: thresh = filters.threshold_otsu(red_enhanced)
    
    mask = red_enhanced > thresh
    mask = morphology.remove_small_objects(mask, min_size=MIN_TUMOR_SIZE)
    mask = morphology.remove_small_holes(mask, area_threshold=HOLE_SIZE)
    
    # Center Priority
    label_img = measure.label(mask)
    regions = measure.regionprops(label_img)
    
    if not regions: return None, None, None
    
    img_h, img_w = raw_red.shape
    center_y, center_x = img_h / 2, img_w / 2
    
    # Pick object closest to image center
    tumor = min(regions, key=lambda r: np.sqrt((r.centroid[0]-center_y)**2 + (r.centroid[1]-center_x)**2))
    
    final_mask = (label_img == tumor.label)
    return final_mask, tumor.centroid, raw_red

#Main Loop
def main():
    if not os.path.exists(OUTPUT_FOLDER): os.makedirs(OUTPUT_FOLDER)
    
    all_files = []
    for ext in ['*.tif', '*.tiff', '*.png']:
        all_files.extend(glob.glob(os.path.join(INPUT_FOLDER, ext)))
        
    wells = {}
    print(f"Scanning {len(all_files)} files...")
    
    for f in all_files:
        fname = os.path.basename(f)
        well_name, dt = parse_filename(fname)
        if well_name:
            if well_name not in wells: wells[well_name] = []
            wells[well_name].append({'path': f, 'time': dt, 'file': fname})
    
    print(f"Found {len(wells)} unique wells.")
    
    results = []

    for well_id, images in wells.items():
        images.sort(key=lambda x: x['time'])
        day0_info = images[0]
        
        print(f"\nProcessing Well: {well_id}")
        
        # Load Day 0
        d0_stack = io.imread(day0_info['path'])
        d0_mask, d0_centroid, d0_raw = get_tumor_mask(d0_stack)
        
        if d0_mask is None:
            print(f"  [Skip] Day 0 segmentation failed for {well_id}")
            continue
            
        d0_area = np.sum(d0_mask)
        
        for img_info in images:
            current_stack = io.imread(img_info['path'])
            curr_mask, curr_centroid, curr_raw = get_tumor_mask(current_stack)
            
            if curr_mask is None: continue
            
            curr_area = np.sum(curr_mask)
            
            d0_mask_aligned = align_mask_by_centroids(
                mask_src=d0_mask, 
                center_src=d0_centroid, 
                shape_dest=curr_mask.shape, 
                center_dest=curr_centroid
            )
            
            # Subtraction
            invasion_mask = curr_mask & (~d0_mask_aligned)
            invasion_area = np.sum(invasion_mask)
            
            # Visualize
            vis_img = cv2.cvtColor(cv2.normalize(curr_raw, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U), cv2.COLOR_GRAY2RGB)
            
            # Draw Day 0 Core (Blue)
            if np.sum(d0_mask_aligned) > 0:
                cnts_d0, _ = cv2.findContours(d0_mask_aligned.astype(np.uint8)*255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(vis_img, cnts_d0, -1, (255, 0, 0), 2)
            
            # Draw Invasion (Yellow Overlay)
            overlay = vis_img.copy()
            overlay[invasion_mask] = (0, 255, 255)
            cv2.addWeighted(overlay, 0.4, vis_img, 0.6, 0, vis_img)
            
            # Draw Current Edge (Red)
            cnts_curr, _ = cv2.findContours(curr_mask.astype(np.uint8)*255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(vis_img, cnts_curr, -1, (0, 0, 255), 1)

            io.imsave(os.path.join(OUTPUT_FOLDER, f"VIS_{img_info['file']}.png"), vis_img, check_contrast=False)
            
            results.append({
                "Well_ID": well_id,
                "Filename": img_info['file'],
                "Hours_Elapsed": (img_info['time'] - day0_info['time']).total_seconds() / 3600,
                "Total_Area_px": curr_area,
                "Invasion_Area_px": invasion_area,
                "Normalized_Invasion_Ratio": invasion_area / d0_area
            })

    if results:
        pd.DataFrame(results).to_csv(os.path.join(OUTPUT_FOLDER, "Invasion_Subtraction_Results.csv"), index=False)
        print("\nDone.")

if __name__ == "__main__":
    main()